# Reproduce the TurboQuant benchmark

Run all cells from any directory in an LMCache checkout. The notebook calls the checked-in benchmark CLI, fails on a non-zero exit status, and checks that all requested presets and the exact tensor shape appear in the output. A CUDA-capable environment with LMCache's test dependencies is required.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
from pathlib import Path
import os
import subprocess
import sys

import torch

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "lmcache").is_dir()
)
BENCHMARK = ROOT / "examples/serde/turboquant/bench_turboquant.py"
assert BENCHMARK.is_file(), BENCHMARK
assert torch.cuda.is_available(), "CUDA is required"
print(
    {
        "root": str(ROOT),
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
    }
)

In [ ]:
# SPDX-License-Identifier: Apache-2.0
layers = int(os.getenv("LMCACHE_TQ_LAYERS", "32"))
tokens = int(os.getenv("LMCACHE_TQ_TOKENS", "8193"))
kv_heads = int(os.getenv("LMCACHE_TQ_KV_HEADS", "8"))
head_dim = int(os.getenv("LMCACHE_TQ_HEAD_DIM", "128"))
warmup = int(os.getenv("LMCACHE_TQ_WARMUP", "5"))
iterations = int(os.getenv("LMCACHE_TQ_ITERS", "20"))
presets = os.getenv(
    "LMCACHE_TQ_PRESETS",
    "turboquant_k8v4 turboquant_4bit_nc turboquant_k3v4_nc turboquant_3bit_nc",
).split()
expected_shape = f"2x{layers}x{tokens}x{kv_heads * head_dim}"
command = [
    sys.executable,
    str(BENCHMARK),
    "--device",
    "cuda",
    "--dtype",
    "bfloat16",
    "--layers",
    str(layers),
    "--tokens",
    str(tokens),
    "--kv-heads",
    str(kv_heads),
    "--head-dim",
    str(head_dim),
    "--warmup",
    str(warmup),
    "--iters",
    str(iterations),
    "--seed",
    "20260904",
    "--presets",
    *presets,
]
print("Running:", " ".join(command))

In [ ]:
# SPDX-License-Identifier: Apache-2.0
env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT) + os.pathsep + env.get("PYTHONPATH", "")
completed = subprocess.run(
    command,
    cwd=ROOT,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(completed.stdout)
completed.check_returncode()
assert expected_shape in completed.stdout, expected_shape
for preset in presets:
    assert preset in completed.stdout, preset
print({"status": "passed", "shape": expected_shape, "presets": presets})